In [ ]:
import os
import torch
import torchaudio
from torchaudio.functional import add_noise
from nnAudio.features.gammatone import Gammatonegram

In [ ]:
SR = 8000
N_FFT = 512
N_BINS = 64
HOP_LENGTH = 128
DURATION_SEC = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SirenGRU(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = torch.nn.GRU(
            input_size=N_BINS,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=False # 실시간 타겟이라 단방향 고정
        )
        self.fc = torch.nn.Linear(128, N_BINS)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        out, _ = self.gru(x)
        return self.sigmoid(self.fc(out))

In [ ]:
if __name__ == "__main__":
    print(f"디바이스: {DEVICE}")
    
    model_path = "/content/siren_gru.pth"

    # 모델 선언 및 학습 가중치 탑재
    model = SirenGRU().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    gt_extractor = Gammatonegram(
        sr=SR, n_fft=N_FFT, n_bins=N_BINS, hop_length=HOP_LENGTH, fmin=100.0, fmax=4000.0
    ).to(DEVICE)

    test_clean_path = "/content/test_siren.wav"
    test_noise_path = "/content/test_siren.wav"

    clean_wave, clean_sr = torchaudio.load(test_clean_path)
    if clean_wave.shape[0] > 1: # 스테레오면 모노로 변환
        clean_wave = torch.mean(clean_wave, dim=0, keepdim=True)
    if clean_sr != SR: # 샘플링 레이트가 다르면 리샘플링
        clean_wave = torchaudio.functional.resample(clean_wave, clean_sr, SR)
    
    noise_wave, noise_sr = torchaudio.load(test_noise_path)
    if noise_wave.shape[0] > 1: # 스테레오면 모노로 변환
        noise_wave = torch.mean(noise_wave, dim=0, keepdim=True)
    if noise_sr != SR: # 샘플링 레이트가 다르면 리샘플링
        noise_wave = torchaudio.functional.resample(noise_wave, noise_sr, SR)

    # 규격에 맞게 2초 길이 강제 조정
    max_len = SR * DURATION_SEC
    if clean_wave.shape[1] > max_len:
        clean_wave = clean_wave[:, :max_len]
    else:
        clean_wave = torch.nn.functional.pad(clean_wave, (0, max_len - clean_wave.shape[1]))
        
    if noise_wave.shape[1] > max_len:
        noise_wave = noise_wave[:, :max_len]
    else:
        noise_wave = torch.nn.functional.pad(noise_wave, (0, max_len - noise_wave.shape[1]))

    # 노이즈 믹싱
    # 사이렌 파워와 소음 파워가 1:1로 섞인 상황 테스트
    target_snr = torch.tensor([0.0]) # 0dB SNR : 사이렌과 소음이 동일한 파워로 섞임
    mixed_wave = add_noise(clean_wave, noise_wave, target_snr)
    
    # 믹싱된 테스트 입력 음원 파일 보관용 저장
    torchaudio.save("test_input_mixed.wav", mixed_wave, SR)

    with torch.no_grad():
        # 혼합 신호의 감마톤 스펙트로그램 추출
        m_gt = gt_extractor(mixed_wave.to(DEVICE)) # Shape: (1, 64, Time)

        # 맨 앞 1차원을 떼고 행렬 전치 수행하여 (Time, 64)로 변환
        X = m_gt[0].transpose(0, 1) 
        # GRU 입력 규격인 (Batch, Time, Features)을 맞추기 위해 0번 축 추가
        X = X.unsqueeze(0)          

        pred_mask = model(X) # 출력 차원: (1, Time, 64)

    # 최종 결과 행렬 복원
    # 배치 차원을 해제하고 다시 주파수 연산을 위해 (64, Time) 형태로 역전치
    mask_2d = pred_mask.squeeze(0).transpose(0, 1) 
    
    # 원본 감마톤 에너지 텐서에 인공지능이 뱉은 필터 마스크(0.0 ~ 1.0)를 직접 곱함
    enhanced_gt = m_gt[0] * mask_2d

    print(f"마스크 최소값 (소음 대역 차단력): {mask_2d.min().item():.4f}")
    print(f"마스크 최대값 (사이렌 대역 보존력): {mask_2d.max().item():.4f}")
    print(">> 최소값이 0에 가깝고 최대값이 1에 가깝게 벌어지면 필터 학습이 성공.")